# Counties Without Public Transit

Day 18 · NULL. US counties with no scheduled stop for any bus, train or ferry, local or long-distance: no stop on a
fixed-route local or regional transit line, no Amtrak stop, no intercity bus or rail terminal, no ferry terminal.
Interactive page (hover a county) plus a still poster for LinkedIn. Run `python scripts\fetch_week3.py day18` first.

## 1 · Settings

In [ ]:
DAY      = 18
PROMPT   = "NULL"
SLUG     = "null-road-only-counties"
NOTEBOOK = "day18_null_road_only_counties.ipynb"
TITLE    = "Counties Without Public Transit"
CREDITS  = ("Transit stops: BTS National Transit Map (NTAD). Amtrak stations and routes: BTS NTAD. Intercity bus, rail "
            "and ferry terminals: BTS Intermodal Passenger Connectivity Database. Counties: US Census Bureau.")

COUNT_AIRPORTS = False     # True = a county with a commercial airport counts as connected

PAGE, INK, SOFT = "#efede8", "#1d1d1f", "#7a7a76"
ROAD, LINKED = "#2f3033", "#d9d6cf"        # asphalt for road-only, pale for connected
RAIL = "#e0442b"                            # Amtrak, signal red
YELLOW = "#f2c230"                          # road-sign yellow, for the counties the train passes without stopping

## 2 · Load

In [ ]:
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from mapkit import poster, publish
from mapkit.paths import PROCESSED, EQUAL_AREA

counties = gpd.read_file(PROCESSED / "counties_connectivity.geojson").to_crs(EQUAL_AREA)
routes = gpd.read_file(PROCESSED / "amtrak_routes.geojson").to_crs(EQUAL_AREA)
stations = gpd.read_file(PROCESSED / "amtrak_stations.geojson").to_crs(EQUAL_AREA)
print(f"{len(counties):,} counties · {len(stations):,} Amtrak stops · {len(routes)} Amtrak route pieces")

## 3 · Analysis, every number on the map comes from here

In [ ]:
links = {
    "fixed-route transit": counties.transit_stops > 0,
    "Amtrak stop":         counties.amtrak_stations > 0,
    "intercity bus":       counties.ipcd_bus_i > 0,
    "rail terminal":       counties.ipcd_mode_rail > 0,
    "ferry":               counties.ipcd_mode_ferry > 0,
}
if COUNT_AIRPORTS:
    links["airport"] = counties.ipcd_mode_air > 0
connected = np.logical_or.reduce(list(links.values()))
counties["road_only"] = ~connected

# Counties the train passes through without a single stop of any kind
rail_stops = stations[stations.StnType.str.upper().str.contains("RAIL", na=False)]
crossed = gpd.sjoin(counties[["geoid", "geometry"]], routes[["geometry"]], predicate="intersects").geoid.unique()
counties["train_passes"] = counties.geoid.isin(crossed)
counties["passed_by"] = counties.road_only & counties.train_passes

area = counties.area
N = len(counties)
STATS = dict(road=int(counties.road_only.sum()), passed=int(counties.passed_by.sum()),
             area_share=area[counties.road_only].sum() / area.sum() * 100)
STATS["share"] = STATS["road"] / N * 100
print(f"{STATS['road']:,} of {N:,} counties in the Lower 48 ({STATS['share']:.0f}%) are road only")
print(f"They cover {STATS['area_share']:.0f}% of the land")
print(f"{STATS['passed']} of them have an Amtrak line running through and nowhere to get on")
print("Counties connected by each kind of link:", {k: int(v.sum()) for k, v in links.items()})
print(counties[counties.road_only].state_abb.value_counts().head(8).to_dict())

## 4 · Map

In [ ]:
F = poster.font
TITLE_F, TEXT_F, TEXT_B = poster.SLAB, poster.BARLOW, poster.BARLOW_BOLD     # Zilla Slab, like a road sign
fig, ax = poster.page(PAGE)
ax.set_position([0.02, 0.21, 0.96, 0.565])
counties[~counties.road_only].plot(ax=ax, color=LINKED, edgecolor=PAGE, lw=0.15, zorder=1)
counties[counties.road_only & ~counties.passed_by].plot(ax=ax, color=ROAD, edgecolor=PAGE, lw=0.15, zorder=2)
counties[counties.passed_by].plot(ax=ax, color=YELLOW, edgecolor=PAGE, lw=0.15, zorder=3)
routes.plot(ax=ax, color=RAIL, lw=1.1, zorder=4)
ax.scatter(rail_stops.geometry.x, rail_stops.geometry.y, s=5, color=PAGE, edgecolor=RAIL, lw=0.6, zorder=5)
b = counties.total_bounds
ax.set_xlim(b[0], b[2]); ax.set_ylim(b[1], b[3]); ax.set_aspect("equal"); ax.set_axis_off()

fig.text(0.06, 0.955, "COUNTIES WITHOUT\nPUBLIC TRANSIT", color=INK, fontproperties=F(TITLE_F, 60), va="top", linespacing=0.95)
fig.text(0.06, 0.835, f"{STATS['road']:,} US counties have no scheduled stop for any bus, train or ferry, local or long-distance.",
         color=INK, fontproperties=F(TEXT_F, 15.5), va="top")
fig.text(0.06, 0.808, f"In {STATS['passed']} of them, an Amtrak line runs straight through without stopping.",
         color=SOFT, fontproperties=F(TEXT_F, 15), va="top")

# legend
lx, ly = 0.06, 0.185
for i, (col, lab, line) in enumerate([(ROAD, "No public transit stop", False), (YELLOW, "None, though an Amtrak line passes through", False),
                                      (LINKED, "At least one stop", False), (RAIL, "Amtrak route", True)]):
    y = ly - i * 0.026
    if line:
        fig.add_artist(plt.Line2D([lx, lx + 0.03], [y + 0.006] * 2, transform=fig.transFigure, color=col, lw=2))
    else:
        fig.add_artist(plt.Rectangle((lx, y), 0.03, 0.013, transform=fig.transFigure, color=col, lw=0))
    fig.text(lx + 0.042, y + 0.0065, lab, color=INK, fontproperties=F(TEXT_F, 12.5), va="center")

# credit, set by hand (Barlow sits taller than the helper expects)
for i, line in enumerate(["US Census counties · Lower 48 · dial-a-ride and airports not counted",
                          "BTS National Transportation Atlas Database: National Transit Map, Amtrak, Intermodal Connectivity"]):
    fig.text(0.06, 0.020 + i * 0.017, line, color=SOFT, fontproperties=F(TEXT_F, 9.5), va="bottom")
fig.text(0.06, 0.057, "Map by Gautami Kasat", color=INK, fontproperties=F(TEXT_B, 12), va="bottom")
plt.show()

## 4b · Interactive page: hover a county to see what it has

In [ ]:
import json

def _r(geom, nd=4):
    return json.loads(json.dumps(geom.__geo_interface__), parse_float=lambda x: round(float(x), nd))

web = counties.to_crs(4326)
status = np.where(counties.passed_by, "passed", np.where(counties.road_only, "none", "some"))
feats = []
for (_, r), st in zip(web.iterrows(), status):
    feats.append({"type": "Feature", "geometry": _r(r.geometry), "properties": {
        "n": r.county_name, "s": r.state_abb, "st": st,
        "stops": int(r.transit_stops), "amtrak": int(r.amtrak_stations),
        "bus": int(r.ipcd_bus_i > 0), "rail": int(r.ipcd_mode_rail > 0), "ferry": int(r.ipcd_mode_ferry > 0),
        "train": int(r.train_passes)}})
web_routes = {"type": "FeatureCollection", "features": [{"type": "Feature", "geometry": _r(g), "properties": {}}
                                                        for g in routes.to_crs(4326).geometry]}
web_stations = {"type": "FeatureCollection", "features": [
    {"type": "Feature", "geometry": _r(r.geometry, 4), "properties": {"n": r.StationName, "t": str(r.StnType).title()}}
    for _, r in stations.to_crs(4326).iterrows()]}

HTML = """<!DOCTYPE html><html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>__TITLE__ · Gautami Kasat</title>
<link href="https://unpkg.com/maplibre-gl@4.7.1/dist/maplibre-gl.css" rel="stylesheet">
<script src="https://unpkg.com/maplibre-gl@4.7.1/dist/maplibre-gl.js"></script>
<link href="https://fonts.googleapis.com/css2?family=Zilla+Slab:wght@700&family=Barlow:wght@400;600&display=swap" rel="stylesheet">
<style>
  html,body{margin:0;height:100%;background:__PAGE__;font-family:Barlow,system-ui,sans-serif;color:__INK__}
  #map{position:absolute;inset:0}
  .panel{position:absolute;top:18px;left:18px;width:340px;max-width:calc(100% - 36px);background:rgba(239,237,232,.94);
         border:1px solid #d4d1ca;border-radius:10px;padding:18px 20px 14px;box-shadow:0 6px 24px rgba(0,0,0,.08)}
  h1{font-family:'Zilla Slab',serif;font-weight:700;font-size:30px;line-height:1.05;margin:0 0 8px;text-transform:uppercase;letter-spacing:.3px}
  .sub{font-size:15px;line-height:1.4;margin:0 0 12px}
  .sub b{font-weight:600}
  .key{display:grid;grid-template-columns:22px 1fr auto;gap:6px 8px;align-items:center;font-size:14px;margin:4px 0 10px}
  .sw{width:20px;height:12px;border-radius:2px}
  .ln{width:20px;height:3px;background:__RAIL__;border-radius:2px}
  .n{font-weight:600;font-variant-numeric:tabular-nums}
  .hint{font-size:12.5px;color:__SOFT__;margin:2px 0 8px}
  details{font-size:12.5px;color:#4a4a47;line-height:1.45;margin-bottom:10px}
  summary{cursor:pointer;font-weight:600;color:__INK__}
  .credit{border-top:1px solid #d4d1ca;padding-top:8px;font-size:11.5px;color:__SOFT__;line-height:1.4}
  .credit b{display:block;color:__INK__;font-size:13px;margin-bottom:1px}
  .maplibregl-popup-content{font-family:Barlow,sans-serif;font-size:13.5px;padding:10px 12px;border-radius:8px;line-height:1.45}
  .pop h3{margin:0 0 4px;font-size:15px}
  .pop .tag{display:inline-block;font-size:11.5px;font-weight:600;padding:1px 6px;border-radius:3px;margin-bottom:5px}
  .pop .row{display:flex;justify-content:space-between;gap:14px}
  .pop .no{color:#9a9893}
  @media (max-width:600px){.panel{width:auto;right:12px;left:12px;top:12px;padding:12px 14px}h1{font-size:24px}details{display:none}}
</style></head><body>
<div id="map"></div>
<div class="panel">
  <h1>__TITLE__</h1>
  <p class="sub">__SUB__</p>
  <div class="key">
    <span class="sw" style="background:__ROAD__"></span><span>No public transit stop</span><span class="n">__N_NONE__</span>
    <span class="sw" style="background:__YELLOW__"></span><span>None, though an Amtrak line passes through</span><span class="n">__N_PASSED__</span>
    <span class="sw" style="background:__LINKED__;border:1px solid #c9c5bc"></span><span>At least one stop</span><span class="n">__N_SOME__</span>
    <span class="ln"></span><span>Amtrak route</span><span></span>
  </div>
  <p class="hint">Hover a county (tap on a phone) to see what it has.</p>
  <details><summary>What counts as public transit</summary>
    A county counts as connected if it has at least one: stop on a scheduled local or regional bus, light rail, subway
    or commuter rail route (National Transit Map); Amtrak station or Amtrak bus stop; intercity bus terminal such as
    Greyhound or Trailways, intercity rail terminal, or ferry terminal (Intermodal Passenger Connectivity Database).
    Dial-a-ride and other on-demand services, school buses and airports are not counted. The National Transit Map only
    includes agencies that publish their schedules, so a few small rural bus services may be missing.</details>
  <div class="credit"><b>Map by Gautami Kasat</b>__CREDITS__</div>
</div>
<script>
const COUNTIES = __COUNTIES__, ROUTES = __ROUTES__, STATIONS = __STATIONS__;
const map = new maplibregl.Map({container:"map", attributionControl:false,
  style:{version:8, sources:{}, layers:[{id:"bg", type:"background", paint:{"background-color":"__PAGE__"}}]},
  bounds:[[-125,24.3],[-66.9,49.4]], fitBoundsOptions:{padding:{top:30,bottom:30,left:window.innerWidth>700?380:20,right:30}}});
map.addControl(new maplibregl.NavigationControl({showCompass:false}), "bottom-right");
map.on("load", () => {
  map.addSource("c", {type:"geojson", data:COUNTIES, generateId:true});
  map.addLayer({id:"fill", type:"fill", source:"c", paint:{"fill-color":["match",["get","st"],"none","__ROAD__","passed","__YELLOW__","__LINKED__"],
    "fill-opacity":["case",["boolean",["feature-state","hover"],false],0.75,1]}});
  map.addLayer({id:"edge", type:"line", source:"c", paint:{"line-color":"__PAGE__","line-width":0.35}});
  map.addSource("r", {type:"geojson", data:ROUTES});
  map.addLayer({id:"rail", type:"line", source:"r", paint:{"line-color":"__RAIL__","line-width":["interpolate",["linear"],["zoom"],3,1.2,8,2.6]}});
  map.addSource("s", {type:"geojson", data:STATIONS});
  map.addLayer({id:"stn", type:"circle", source:"s", minzoom:5, paint:{"circle-radius":3,"circle-color":"__PAGE__","circle-stroke-color":"__RAIL__","circle-stroke-width":1.2}});
  map.addLayer({id:"hl", type:"line", source:"c", paint:{"line-color":"__INK__","line-width":["case",["boolean",["feature-state","hover"],false],1.6,0]}});
  const pop = new maplibregl.Popup({closeButton:false, closeOnClick:false, offset:10});
  let hov = null;
  const yes = (v, txt) => v ? `<span>${txt}</span><b>yes</b>` : `<span class="no">${txt}</span><span class="no">none</span>`;
  map.on("mousemove", "fill", e => {
    const f = e.features[0], p = f.properties;
    if (hov !== null) map.setFeatureState({source:"c", id:hov}, {hover:false});
    hov = f.id; map.setFeatureState({source:"c", id:hov}, {hover:true});
    map.getCanvas().style.cursor = "pointer";
    const tag = p.st === "some" ? ["#d9d6cf","#1d1d1f","Has public transit"] :
                p.st === "passed" ? ["__YELLOW__","#1d1d1f","No stop, Amtrak passes through"] : ["__ROAD__","#fff","No public transit stop"];
    pop.setLngLat(e.lngLat).setHTML(`<div class="pop"><h3>${p.n}, ${p.s}</h3>
      <span class="tag" style="background:${tag[0]};color:${tag[1]}">${tag[2]}</span>
      <div class="row"><span class="${p.stops?'':'no'}">Local and regional transit stops</span><b class="${p.stops?'':'no'}">${p.stops.toLocaleString()}</b></div>
      <div class="row"><span class="${p.amtrak?'':'no'}">Amtrak stops</span><b class="${p.amtrak?'':'no'}">${p.amtrak}</b></div>
      <div class="row">${yes(p.bus,"Intercity bus terminal")}</div>
      <div class="row">${yes(p.ferry,"Ferry terminal")}</div></div>`).addTo(map);
  });
  map.on("mouseleave", "fill", () => { if (hov !== null) map.setFeatureState({source:"c", id:hov}, {hover:false}); hov = null;
                                      pop.remove(); map.getCanvas().style.cursor = ""; });
  const sp = new maplibregl.Popup({closeButton:false, offset:8});
  map.on("mouseenter", "stn", e => { const p = e.features[0].properties;
    sp.setLngLat(e.features[0].geometry.coordinates).setHTML(`<b>${p.n}</b><br>Amtrak ${p.t.toLowerCase()} stop`).addTo(map); });
  map.on("mouseleave", "stn", () => sp.remove());
});
</script></body></html>"""

N_SOME = int((~counties.road_only).sum())
SUB_WEB = (f"<b>{STATS['road']:,} of {N:,}</b> counties in the Lower 48 have no scheduled stop for any bus, train or ferry, "
           f"local or long-distance. In <b>{STATS['passed']}</b> of them, an Amtrak line runs straight through without stopping.")
fill = {"__TITLE__": TITLE, "__SUB__": SUB_WEB, "__PAGE__": PAGE, "__INK__": INK, "__SOFT__": SOFT, "__ROAD__": ROAD,
        "__LINKED__": LINKED, "__RAIL__": RAIL, "__YELLOW__": YELLOW,
        "__N_NONE__": f"{STATS['road'] - STATS['passed']:,}", "__N_PASSED__": f"{STATS['passed']:,}", "__N_SOME__": f"{N_SOME:,}",
        "__CREDITS__": ("Transit stops: BTS National Transit Map. Amtrak stations and routes, intercity terminals: BTS National "
                        "Transportation Atlas Database. Counties: US Census Bureau. Lower 48."),
        "__COUNTIES__": json.dumps({"type": "FeatureCollection", "features": feats}, separators=(",", ":")),
        "__ROUTES__": json.dumps(web_routes, separators=(",", ":")),
        "__STATIONS__": json.dumps(web_stations, separators=(",", ":"))}
page = HTML
for k, v in fill.items():
    page = page.replace(k, v)
print(f"page size {len(page) / 1e6:.1f} MB")

## 5 · Export: the interactive page (live map + gallery), and the still poster for LinkedIn

In [ ]:
from mapkit.paths import OUTPUT
publish.save(page, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE,
             subtitle=f"{STATS['road']:,} US counties with no scheduled bus, train or ferry stop; in {STATS['passed']}, Amtrak runs through without stopping",
             series="city", tool="Python + MapLibre", notebook=NOTEBOOK, credits=CREDITS)
fig.savefig(OUTPUT / f"day{DAY:02d}_{SLUG}_poster.png", dpi=100, facecolor=PAGE)
print("poster     output/" + f"day{DAY:02d}_{SLUG}_poster.png")